# Phase 3: entity resolution
Goal: find records that describe the same company under different IDs.

- Matching unit: source record (`entity_id`).
- `canonical_id` (OpenSanctions' grouping): answer key for Phase 4 only. Never used to match.
- Snapshot pinned to 20261030005356-xkx (same as Phase 2) for a clean before/after.

Steps: records, clean names, ID matches, blocking, scoring, combine, grouping, output.

### Step 1: records table
Notebook 13's pivot, grouped by `entity_id` instead of `canonical_id`. One row per source record, before any merging.
- Schemas: Company, Organization, LegalEntity, PublicBody. People excluded.
- Names: name, alias, previousName. weakAlias excluded (unreliable).
- IDs: INN, OGRN, registration number, tax number, LEI.

In [0]:
%python
# dbutils.library.restartPython()
%pip install unidecode
%pip install anyascii

In [0]:
%python

dbutils.library.restartPython()


In [0]:
-- English to latin translation
-- %python
-- from unidecode import unidecode
-- from anyascii import anyascii

-- list=[
--     "медстандарт", "товариство", "société", "阿克苏弘东汽车销售", "печать"
-- ]

-- for each in list:
--   print(each, "-> ",unidecode(each)+" Made with unidecode")
--   print(each, "-> ",anyascii(each)+" Made with anyascii")

In [0]:
-- The query same as 13_legal_entity's notebook source but rather than grouping by canonical_ids(which means those were already identified similar entities and processed by opensanctions), we group by entity_ids so we can process everything from ground.

--Why legal entity because we only wanna include records and its linked details which are not a person that means it could either one of ('Company','Organization','LegalEntity','PublicBody')

CREATE OR REPLACE TABLE pep_dev.eval.er_records 
AS
SELECT  
    psr.entity_id,
    -- canonical_id = OpenSanctions' grouping. Answer key for Phase 4 only; never used to match.
    MAX(psr.canonical_id) AS canonical_id,
  
    collect_set(value)  FILTER( WHERE prop IN ('name', 'alias', 'previousName')) AS names,
    
    coalesce (
        MAX(CASE WHEN prop = 'jurisdiction' THEN value END),
        MAX(CASE WHEN prop = 'mainCountry' THEN value END),
        MAX(CASE WHEN prop = 'country' THEN value END)
    ) AS country,

    collect_set(value) FILTER(WHERE prop = 'innCode') AS inn_codes ,
    collect_set(value) FILTER(WHERE prop = 'ogrnCode') AS ogrn_codes,
    collect_set(value) FILTER(WHERE prop = 'registrationNumber') AS registration_numbers,
    collect_set(value) FILTER(WHERE prop = 'taxNumber') AS tax_numbers,
    collect_set(value) FILTER(WHERE prop = 'leiCode') AS lei_codes,
   
    collect_set(dataset) AS datasets,
    MAX(_snapshot_version) AS _last_seen_snapshot
    
FROM 
pep_dev.bronze.statements AS pbs
JOIN pep_dev.silver.referent AS psr
ON psr.entity_id = pbs.entity_id  
WHERE SCHEMA IN ('Company','Organization','LegalEntity','PublicBody')
AND psr.canonical_id NOT IN (SELECT canonical_id FROM pep_dev.silver.referent WHERE referent_kind = 'Person')
AND _snapshot_version='20261030005356-xkx'
-- PINNED to the Phase 2 snapshot for a clean before/after match in development phase. Gonna Switch back to the subquery below in Phase 6.

-- (
-- SELECT MAX(snapshot_version) FROM pep_dev.bronze.snapshot_version_control)

GROUP BY psr.entity_id

In [0]:
SELECT COUNT(*), COUNT(DISTINCT entity_id) FROM pep_dev.eval.er_records;

WITH groups  AS (
SELECT canonical_id, 
       count(*) as n_records 
 FROM pep_dev.eval.er_records 
GROUP BY canonical_id

)
SELECT  
    SUM(n_records)  filter(WHERE n_records>1)  
    as records_in_multi_groups,
    max(n_records) largest_group,
    count_if(n_records>1) multi_record_groups,
    count(*) all_groups,
    SUM(n_records*(n_records-1)/2) AS possible_answer_key_pairs

  FROM groups ;

In [0]:
SELECT DISTINCT(_last_seen_snapshot) FROM pep_dev.eval.er_records

In [0]:
SELECT count(*),count_if(size(names) = 0) FROM pep_dev.eval.er_records
WHERE size(inn_codes)>0
        OR
      size(ogrn_codes)>0  
      OR
      size(registration_numbers)>0
        OR
      size(tax_numbers)>0  
      OR
      size(lei_codes) > 0

In [0]:
SELECT * FROM pep_dev.eval.er_records

**PHASE 2**

In [0]:
CREATE OR REPLACE TABLE pep_dev.eval.er_names
AS
SELECT 
    entity_id,
    country,
    explode(names) as raw_name
FROM pep_dev.eval.er_records;

### Step 2d: legal forms list
Legal forms (ООО, LLC, 有限公司) say what *type* a company is, not *which* one.
Thousands of unrelated companies share them, and one company writes them several ways
(ооо ромашка / общество с ограниченной ответственностью ромашка / romashka llc).
Removing them leaves the part that identifies the company.

Stored as data, one row per form. To cover a new form, add a row; no query changes.

In [0]:
-- Legal forms removed from company names before matching.
-- Written in cleaned form (lowercase, no punctuation, dots deleted), because they run after step 2c:
--   s.a. -> sa,  sp. z o.o. -> sp z oo,  co., ltd. -> co ltd
-- word_boundary TRUE : removed only as a whole word ("co" in "abc co", never inside "coal")
-- word_boundary FALSE: Chinese/Japanese, written without spaces, so no word edges exist
-- Row order doesn't matter: the legal_patterns view sorts longest first.
-- Left out on purpose:
--   group, holding, bank, trading       -> carry meaning
--   учреждение, государственное, ...    -> state bodies, not company forms
--   as, spa, sas, kg                    -> also ordinary words or names
-- CREATE OR REPLACE empties the table, so rerunning this cell doesn't duplicate rows.


CREATE OR REPLACE TABLE pep_dev.eval.ref_legal_forms (
    pattern        STRING   COMMENT 'Legal form in cleaned form (lowercase, no punctuation). May hold small regex parts for spelling variants.',
        -- Russian, transliterated. Small regex parts cover spelling variants:
    --   (h)? = optional h (obshchestvo / obschestvo),  [ijy] = one of i, j, y,  (yu|ju|iu) = one of three endings
    language       STRING   COMMENT 'Origin of the form',
    word_boundary  BOOLEAN  COMMENT 'FALSE for scripts written without spaces (Chinese, Japanese)'
);

INSERT INTO pep_dev.eval.ref_legal_forms VALUES
    -- Russian
    ('общество с ограниченной ответственностью',            'ru',      TRUE),
    ('непубличное акционерное общество',                    'ru',      TRUE),
    ('публичное акционерное общество',                      'ru',      TRUE),
    ('открытое акционерное общество',                       'ru',      TRUE),
    ('закрытое акционерное общество',                       'ru',      TRUE),
    ('акционерное общество',                                'ru',      TRUE),
    ('федеральное государственное унитарное предприятие',   'ru',      TRUE),
    ('государственное унитарное предприятие',               'ru',      TRUE),
    ('муниципальное унитарное предприятие',                 'ru',      TRUE),
    ('ооо', 'ru', TRUE), ('пао', 'ru', TRUE), ('оао', 'ru', TRUE),
    ('зао', 'ru', TRUE), ('ао', 'ru', TRUE),
    ('фгуп', 'ru', TRUE), ('гуп', 'ru', TRUE), ('муп', 'ru', TRUE),
    ('лимитед', 'ru', TRUE), ('лтд', 'ru', TRUE),

    -- Ukrainian
    ('товариство з обмеженою відповідальністю',             'uk',      TRUE),
    ('приватне акціонерне товариство',                      'uk',      TRUE),
    ('публічне акціонерне товариство',                      'uk',      TRUE),
    ('відкрите акціонерне товариство',                      'uk',      TRUE),
    ('закрите акціонерне товариство',                       'uk',      TRUE),
    ('акціонерне товариство',                               'uk',      TRUE),
    ('тов', 'uk', TRUE), ('прат', 'uk', TRUE), ('пат', 'uk', TRUE),
    ('ат', 'uk', TRUE), ('лімітед', 'uk', TRUE),

    -- Russian, transliterated (spelling varies by source)
    ('obs(h)?chestvo s ogranichenno[ijy] otvetstvennost(yu|ju|iu)', 'ru-latn', TRUE),
    ('otkrytoe aktsionernoe obs(h)?chestvo',                'ru-latn', TRUE),
    ('zakrytoe aktsionernoe obs(h)?chestvo',                'ru-latn', TRUE),
    ('publichnoe aktsionernoe obs(h)?chestvo',              'ru-latn', TRUE),
    ('aktsionernoe obs(h)?chestvo',                         'ru-latn', TRUE),
    ('ooo', 'ru-latn', TRUE), ('oao', 'ru-latn', TRUE), ('zao', 'ru-latn', TRUE),
    ('pao', 'ru-latn', TRUE), ('ao', 'ru-latn', TRUE),

    -- Ukrainian, transliterated
    ('tovarystvo z obmezheno(iu|yu|ju) vidpovidalnist(iu|yu|ju)', 'uk-latn', TRUE),
    ('pryvatne aktsionerne tovarystvo',                     'uk-latn', TRUE),
    ('publichne aktsionerne tovarystvo',                    'uk-latn', TRUE),
    ('vidkryte aktsionerne tovarystvo',                     'uk-latn', TRUE),
    ('zakryte aktsionerne tovarystvo',                      'uk-latn', TRUE),
    ('aktsionerne tovarystvo',                              'uk-latn', TRUE),
    ('tov', 'uk-latn', TRUE), ('prat', 'uk-latn', TRUE),

    -- English
    ('limited liability company',     'en', TRUE),
    ('limited liability partnership', 'en', TRUE),
    ('public joint stock company',    'en', TRUE),
    ('open joint stock company',      'en', TRUE),
    ('closed joint stock company',    'en', TRUE),
    ('joint stock company',           'en', TRUE),
    ('public company limited',        'en', TRUE),
    ('private limited',               'en', TRUE),
    ('company limited',               'en', TRUE),
    ('co limited',                    'en', TRUE),
    ('co ltd',                        'en', TRUE),
    ('pvt ltd',                       'en', TRUE),
    ('limited', 'en', TRUE), ('ltd', 'en', TRUE),
    ('llc', 'en', TRUE), ('llp', 'en', TRUE), ('lp', 'en', TRUE),
    ('plc', 'en', TRUE), ('pllc', 'en', TRUE),
    ('incorporated', 'en', TRUE), ('inc', 'en', TRUE),
    ('corporation', 'en', TRUE), ('corp', 'en', TRUE),
    ('company', 'en', TRUE), ('co', 'en', TRUE),
    ('jsc', 'en', TRUE), ('pjsc', 'en', TRUE), ('ojsc', 'en', TRUE), ('cjsc', 'en', TRUE),

    -- Other countries
    ('gmbh co kg', 'de', TRUE), ('gmbh', 'de', TRUE), ('ag', 'de', TRUE),
    ('s de rl de cv', 'mx', TRUE), ('sa de cv', 'mx', TRUE), ('cv', 'mx', TRUE),
    ('sa', 'intl', TRUE), ('sarl', 'fr', TRUE), ('srl', 'it', TRUE), ('sl', 'es', TRUE),
    ('ltda', 'br', TRUE), ('limitada', 'br', TRUE), ('eireli', 'br', TRUE),
    ('bv', 'nl', TRUE), ('nv', 'nl', TRUE),
    ('sp z oo', 'pl', TRUE), ('sro', 'cz', TRUE),
    ('doo', 'si', TRUE), ('dd', 'si', TRUE), ('sp', 'si', TRUE),
    ('kft', 'hu', TRUE), ('zrt', 'hu', TRUE), ('nyrt', 'hu', TRUE), ('rt', 'hu', TRUE),
    ('oyj', 'fi', TRUE), ('oy', 'fi', TRUE), ('ab', 'se', TRUE),
    ('anonim sirketi', 'tr', TRUE), ('limited sirketi', 'tr', TRUE),
    ('ltd sti', 'tr', TRUE), ('sti', 'tr', TRUE),
    ('pte', 'sg', TRUE), ('sdn bhd', 'my', TRUE), ('bhd', 'my', TRUE),
    ('pvt', 'in', TRUE), ('pty', 'au', TRUE), ('pt', 'id', TRUE), ('tbk', 'id', TRUE),
    ('fz llc', 'ae', TRUE), ('fzco', 'ae', TRUE), ('fzc', 'ae', TRUE),
    ('fze', 'ae', TRUE), ('dmcc', 'ae', TRUE),
    ('შპს', 'ge', TRUE),
    ('المحدودة', 'ar', TRUE), ('شركة', 'ar', TRUE), ('شرکت', 'fa', TRUE),

    -- Chinese and Japanese (no spaces, so no word boundary)
    ('股份有限公司', 'zh', FALSE), ('有限责任公司', 'zh', FALSE),
    ('有限公司', 'zh', FALSE), ('有限合伙', 'zh', FALSE),
    ('株式会社', 'ja', FALSE), ('有限会社', 'ja', FALSE),

    -- Dataset noise (note attached to Chinese mining records)
    ('historical annual correction situation', 'noise', TRUE),
    ('历史年度', 'noise', FALSE), ('更正情况', 'noise', FALSE);

### Step 2d: list to regex
Turns the list into two regex strings:
- `bounded_pattern`: forms removed as whole words only
- `unbounded_pattern`: Chinese/Japanese forms, removed wherever they appear

In [0]:

-- One row, two strings, e.g. 'limited liability company|...|co'.
-- A view, not a table: always reads the current list, so a new row in ref_legal_forms is picked up.

CREATE OR REPLACE VIEW pep_dev.eval.legal_patterns 
AS
SELECT
    -- collect_list : rows -> one array
    -- array_sort   : longest first. The regex takes the first option that fits, so "limited"
    --                before "limited liability company" would leave "liability company" behind.
    -- array_join   : array -> one string, options separated by | ("or")
    array_join(
        array_sort(
                collect_list(pattern) FILTER(WHERE  word_boundary),
                (a, b) -> length(b) - length(a)            -- negative = a first, so the longer one goes first
        ), 
    '|' ) as bounded_pattern,
    -- Same steps for forms written without spaces

    array_join(    
        array_sort(
                collect_list(pattern) FILTER(WHERE  NOT word_boundary),
                (a, b) -> length(b) - length(a)
        ),
    '|')
      
    
    as unbounded_pattern

FROM pep_dev.eval.ref_legal_forms;
-- Check: one row; first option is the longest form, last is the shortest
SELECT * FROM pep_dev.eval.legal_patterns;

### Steps 2c-2d: clean names
| Stage | Example |
|---|---|
| raw_name | ООО «Ромашка-Плюс» |
| cleaned_string | ооо ромашка плюс |
| no_legal_name | ромашка плюс |

In [0]:
-- Regex patterns are raw strings (r'...'): Databricks drops a single backslash in a normal string.

CREATE OR REPLACE TABLE pep_dev.eval.er_names_clean 
USING DELTA
AS
WITH cleaned as (

SELECT 
entity_id,
country,
raw_name,
        -- Read inside out: each layer works on the result of the one inside it.

        -- Layers, read inside out:
        --   1st: apostrophes and dots deleted, no space (об'єднання stays one word, s.a. -> sa)
        --   2nd: any punctuation (P) or symbol (S) -> space, not deleted (joint-stock -> joint stock)
        --   3rd: \s+ = run of spaces -> one space (2nd leaves "co  ltd"; the legal-form list uses single spaces)
        --   4th: trim spaces at both ends

 trim(                                                                -- 4th
            regexp_replace(                                           -- 3rd
                regexp_replace(                                       -- 2nd
                    regexp_replace(lower(raw_name), r"['’‘ʼ.]", ''),  -- 1st
                    r'[\p{P}\p{S}]', ' '                              -- 2nd: pattern, replacement
                ),
                r'\s+', ' '                                           -- 3rd: pattern, replacement
            )
        ) AS cleaned_string
    FROM pep_dev.eval.er_names
), 

-- Remove legal forms (list in ref_legal_forms, built into patterns by the legal_patterns view)
no_legal AS (
     SELECT entity_id, country,raw_name,
         trim(
            regexp_replace(
            regexp_replace(
            regexp_replace(cleaned_string, concat(r'(?U)\b(',bounded_pattern,r')\b'), ' ')  -- (?U): Cyrillic and other alphabets count as letters
                                                                                            -- \b..\b : whole words only ("co", never inside "coal")
            ,unbounded_pattern ,                                                       -- Chinese/Japanese: no \b, since no spaces = no word edges
            
            ' '   
            ), 
             r'\s+',' ')
         ) AS clean_name
  FROM cleaned
  CROSS JOIN  pep_dev.eval.legal_patterns    -- one row, so every name gets both patterns; row count unchanged
),



words as (
SELECT entity_id,
       country,
       explode(split(clean_name,' ') ) as word
FROM no_legal
)
SELECT 
entity_id,country,min(raw_name)AS raw_name,clean_name
FROM no_legal
WHERE length(clean_name)>=2
GROUP BY entity_id,country,clean_name





In [0]:
SELECT * FROM pep_dev.eval.er_names_clean;

In [0]:
-- Check: records with no usable name left (name was only a legal form, or under 2 characters).
-- 21 name sets. These can still match by ID in Step 3.
SELECT  DISTINCT(er.names) 
FROM
pep_dev.eval.er_records AS er
LEFT ANTI JOIN pep_dev.eval.er_names_clean AS en
ON er.entity_id=en.entity_id;

### Step 2e: transliterate names to Latin

Output: `eval.er_names_latin`, 725,776 rows (one per record and Latin name).

| # | Layer | Why |
|---|---|---|
| 1 | `to_latin(clean_name)` (anyascii UDF) | Cyrillic, Chinese, Arabic, accents to ASCII |
| 2 | `lower()` | anyascii keeps capitals on Chinese |
| 3 | delete `'` and `` ` `` | Russian soft/hard signs, Arabic hamza and ayn |
| 4 | `[^a-z0-9 ]` to space | safety net for anything else |
| 5 | collapse spaces, trim | tidy |
| 6 | GROUP BY entity_id, country, latin_name | same name in two scripts becomes one row (14,420 removed) |

anyascii over unidecode: Chinese comes out as one word, not separate syllables.
Known limit: Arabic loses short vowels, so it won't name-match English spellings.

In [0]:
%python
from anyascii import anyascii
def to_latin(text: str|None) -> str|None:
        if text is  None:
            return None
        return anyascii(text)
    
spark.udf.register("to_latin", to_latin)


In [0]:
CREATE OR REPLACE TABLE pep_dev.eval.er_names_latin AS
WITH latin as(
SELECT
  entity_id,
  country,
  raw_name,
  clean_name,
  trim(
    regexp_replace(
      regexp_replace(
        regexp_replace(
          lower(to_latin(clean_name)),   -- 1-2: transliterate, then lowercase
          r"['`]", ''),                  -- 3: delete apostrophes and backticks created by anyascii
        r'[^a-z0-9 ]', ' '),             -- 4: anything else left becomes a space (safety net)
      r'\s+', ' ')                       -- 5: collapse repeated spaces
  ) AS latin_name
FROM pep_dev.eval.er_names_clean)

SELECT 
  entity_id, 
  country, 
  min(raw_name) AS raw_name,
  min(clean_name) AS clean_name,
  latin_name
FROM latin
GROUP BY entity_id, country, latin_name;



In [0]:
SELECT entity_id,latin_name FROM pep_dev.eval.er_names_latin
GROUP BY entity_id,latin_name
HAVING count(*)>1;

SELECT
  (SELECT count(DISTINCT entity_id) FROM pep_dev.eval.er_names_clean) AS clean_records,
  (SELECT count(DISTINCT entity_id) FROM pep_dev.eval.er_names_latin) AS latin_records;